In [1]:
import torch
from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

local_model_path = "/active-data/datasets/models/InstaDeepAI/nucleotide-transformer-2.5b-multi-species"

tokenizer = AutoTokenizer.from_pretrained(local_model_path, local_files_only=True)
nt_model = AutoModel.from_pretrained(local_model_path, local_files_only=True).to(device).eval()

/home/zhongshitong/miniconda3/envs/nt_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 518/518 [00:00<00:00, 26927.55it/s]
[transformers] EsmModel LOAD REPORT from: /active-data/datasets/models/InstaDeepAI/nucleotide-transformer-2.5b-multi-species
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect iden

In [2]:
# test
dna_seq = "ATGGCCGTTGACCGATCGA"

inputs = tokenizer(
    dna_seq,
    return_tensors="pt",
    truncation=True,
    max_length=1000
).to(device)

with torch.no_grad():
    out = nt_model(**inputs)

seq_emb = out.last_hidden_state[:, 0, :].cpu().numpy()

input_len = inputs["input_ids"][0].shape[0]
token_emb = out.last_hidden_state[0, 1 : input_len - 1, :].cpu()
print(token_emb)

tensor([[-0.6838,  0.3201,  0.0518,  ..., -0.3909,  0.4533,  0.5406],
        [-0.4465,  0.2953,  0.0830,  ..., -0.2398,  0.3330,  0.4083],
        [-0.9728, -0.1464,  0.2697,  ..., -0.6197,  0.4571,  0.6851]])


In [3]:
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

def sequence_size(seq_id):
    acc_n, contig, start_site, end_site = seq_id.split('-')
    return int(end_site) - int(start_site)

def sequence_extract(acc_n, contig, start_site=None, end_site=None):
    gbff_path = f"/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff"
    for record in SeqIO.parse(gbff_path, "genbank"):
        if record.id != contig:
            continue
        if start_site and end_site:
            return str(record.seq)[int(start_site):int(end_site)]
        else:
            return str(record.seq)

base_folder = f'/active-data/analysis_results/chr_pla/genus'
genus_name = 'Escherichia'

replicon_data = pd.read_csv(f'{base_folder}/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
replicon_data.rename(columns={'accession': 'sequence'}, inplace=True)

chr_frag = pd.read_csv(f'{base_folder}/kmer_chr_frag_samples/{genus_name}/chromosome_fragment_data.tsv', sep='\t')
chr_frag['category-pident_90'] = 'chromosome fragment'
chr_frag['size'] = chr_frag['sequence'].apply(sequence_size)

all_data = pd.concat([replicon_data[['sequence', 'size', 'category-pident_90']], chr_frag[['sequence', 'size', 'category-pident_90']]], ignore_index=True)

In [4]:
import os
from Bio import SeqIO

random_seed = 42
cut_len = 6000
target_dir = f'{base_folder}/embedding_vector/{genus_name}/nt_mean'
vector_dir = f'{target_dir}/random_seed_{random_seed}'
os.makedirs(vector_dir, exist_ok=True)
categories = ['intermediate replicon', 'typical plasmid', 'chromosome fragment', 'typical chromosome']
sample_info = []
for cat in categories:
    try:
        sample = all_data[all_data["category-pident_90"]==cat].sample(n=200, random_state=random_seed).copy().sample(frac=1, random_state=random_seed+1).reset_index(drop=True)
    except:
        sample = all_data[all_data["category-pident_90"]==cat].copy().sample(frac=1, random_state=random_seed+1).reset_index(drop=True)

    with tqdm(total = len(sample), desc=f'{genus_name}:{cat}:{random_seed}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for idx, row in sample.iterrows():
            info = row['sequence'].split('-')
            if len(info) == 2:
                acc_n, contig = info
                dna_seq = sequence_extract(acc_n, contig)
                start_site, end_site = 0, len(dna_seq)
            elif len(info) == 4:
                acc_n, contig, start_site, end_site = info
                dna_seq = sequence_extract(acc_n, contig, start_site, end_site)
            if len(dna_seq) > cut_len:
                n = len(dna_seq) - cut_len
                random.seed(idx + len(dna_seq))
                cut_start = random.randrange(0, n)
                cut_end = cut_start + cut_len
            else:
                cut_start, cut_end = 0, len(dna_seq)
                
            sequence = dna_seq[cut_start:cut_end]
            inputs = tokenizer(
                dna_seq,
                return_tensors="pt",
                truncation=True,
                max_length=1000
            ).to(device)
            
            with torch.no_grad():
                out = nt_model(**inputs)
            
            seq_emb = out.last_hidden_state[:, 0, :].cpu().numpy()
            
            input_len = inputs["input_ids"][0].shape[0]
            token_emb = out.last_hidden_state[0, 1 : input_len - 1, :].cpu()
            
            avg_1d = token_emb.mean(dim=0)
            avg_np = avg_1d.detach().cpu().to(torch.float32).numpy()
            
            os.chdir(vector_dir)
            np.save(f"{row['sequence']}.npy", avg_np)
            sample_info.append(pd.DataFrame([dict(row) | {'cut_start': cut_start, 'cut_end': cut_end, 'length': len(sequence), 'full_size': len(sequence)==len(dna_seq)}]))
            pbar.update(1)

sample_info = pd.concat(sample_info, ignore_index=True)
sample_info.to_csv(f'{vector_dir}_samples_info.tsv', sep='\t', index=False)

Escherichia:typical chromosome:42: 100%|████████████████████████████| 200/200 [11:35<00:00, 3.48s/B]
